In [1]:
import sys
sys.path.append('../lib') 
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import Window

In [2]:
from load_netmob import Load_NetMob

df = Load_NetMob([1, 3, 2])

initializing spark session ...
merging datasets ...
merging with trips ...
merging with individuals ...
merging with weather ...


In [3]:
df.remove_blurring()

Remove blurring effect ...


In [4]:
df.data.show(5)

+-------+-----------+----------+-----+-------------------+-----+-------------------+------------+----+-----------+-----------------+--------------------+--------+----------+--------+------------+------------+------------+------------+------+------+--------+--------+--------+---------+-----------+---------+------------+---------------+----------------+----------------+--------------------+--------------+---------------+-----------------+-----------------+-----------------+-----------+-----------+-------------------+--------------------+-------------------+--------------------+-------------------+-------------------+------+------------+---+-------+------------+----+-------------+-----------+-------------+----+----+----+----+
|     ID|   LATITUDE| LONGITUDE|SPEED|     LOCAL DATETIME|VALID|       UTC DATETIME|      fclass|name|index_right| distance_to_road|                 KEY| Day_EMG|  Date_EMG|Day_Type|ID_Trip_Days|Type_Trip_OD|Code_INSEE_O|Code_INSEE_D|Zone_O|Zone_D|  Time_O|  Time_D|D

In [5]:
df.data.count()

4335029

## Add distance (walked in each road in KM) feature

In [6]:
data_with_ranks = df.data.withColumn(
      'rank_col', 
      row_number().over(
        Window.partitionBy('index_right', 'KEY').orderBy('LOCAL DATETIME')
      )
    )

In [7]:
# Step 2: Self join to calculate the distance between consecutive rows
data_with_distances = data_with_ranks.alias('t1').join(
  data_with_ranks.alias('t2'),
  (col('t1.KEY') == col('t2.KEY')) & 
  (col('t1.index_right') == col('t2.index_right')) &
  (col('t1.rank_col') == col('t2.rank_col') + 1),
  'left'
).select(
    col('t1.*'),
    when(col('t2.LATITUDE').isNotNull() & col('t2.LONGITUDE').isNotNull(),
         expr("""
             6371 * 2 * ASIN(SQRT(
                 POW(SIN(RADIANS(t2.LATITUDE - t1.LATITUDE) / 2), 2) +
                 COS(RADIANS(t1.LATITUDE)) * COS(RADIANS(t2.LATITUDE)) *
                 POW(SIN(RADIANS(t2.LONGITUDE - t1.LONGITUDE) / 2), 2)
             ))
         """))
    .otherwise(lit(0.0)).alias('distance_km')
)

In [8]:
data_with_distances.persist()

DataFrame[ID: string, LATITUDE: double, LONGITUDE: double, SPEED: float, LOCAL DATETIME: timestamp, VALID: string, UTC DATETIME: timestamp, fclass: string, name: string, index_right: string, distance_to_road: double, KEY: string, Day_EMG: string, Date_EMG: date, Day_Type: string, ID_Trip_Days: int, Type_Trip_OD: string, Code_INSEE_O: string, Code_INSEE_D: string, Zone_O: string, Zone_D: string, Time_O: string, Time_D: string, Duration: float, Purpose_O: string, Purpose_D: string, Main_Mode: string, Weight_Day: double, Day_EMG_encoded: int, Date_EMG_encoded: int, Day_Type_encoded: int, Type_Trip_OD_encoded: int, Zone_O_encoded: int, Zone_D _encoded: int, Purpose_O_encoded: int, Purpose_D_encoded: int, Main_Mode_encoded: int, Time_O_Hour: int, Time_D_Hour: int, Time_O_sin: double, Time_O_cos: double, Time_D_sin: double, Time_D_cos: double, Trip_start: timestamp, Trip_end: timestamp, NB_CAR: int, WEIGHT_INDIV: double, AGE: int, PRO_CAT: int, NBPERS_HOUSE: int, BIKE: int, ELECT_SCOOTER: in

In [9]:
data_with_distances.show(5)

+-------+-----------+----------+-----+-------------------+-----+-------------------+------------+--------------------+-----------+------------------+--------------------+--------+----------+--------+------------+------------+------------+------------+------+------+--------+--------+--------+-----------+-----------+---------+------------+---------------+----------------+----------------+--------------------+--------------+---------------+-----------------+-----------------+-----------------+-----------+-----------+-------------------+--------------------+-------------------+--------------------+-------------------+-------------------+------+------------+---+-------+------------+----+-------------+-----------+-------------+----+----+----+----+--------+--------------------+
|     ID|   LATITUDE| LONGITUDE|SPEED|     LOCAL DATETIME|VALID|       UTC DATETIME|      fclass|                name|index_right|  distance_to_road|                 KEY| Day_EMG|  Date_EMG|Day_Type|ID_Trip_Days|Type_T

In [13]:
sample = data_with_distances.filter(col('index_right') == '201733').toPandas()

In [ ]:
sample.to_csv('../data/201733.csv')

In [11]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="fclass", outputCol="fclass_encoded")

data_with_distances_1 = indexer.fit(data_with_distances).transform(data_with_distances)

In [12]:
data_with_distances_1.show(5)

+-------+-----------+----------+-----+-------------------+-----+-------------------+------------+--------------------+-----------+------------------+--------------------+--------+----------+--------+------------+------------+------------+------------+------+------+--------+--------+--------+-----------+-----------+---------+------------+---------------+----------------+----------------+--------------------+--------------+---------------+-----------------+-----------------+-----------------+-----------+-----------+-------------------+--------------------+-------------------+--------------------+-------------------+-------------------+------+------------+---+-------+------------+----+-------------+-----------+-------------+----+----+----+----+--------+--------------------+--------------+
|     ID|   LATITUDE| LONGITUDE|SPEED|     LOCAL DATETIME|VALID|       UTC DATETIME|      fclass|                name|index_right|  distance_to_road|                 KEY| Day_EMG|  Date_EMG|Day_Type|ID_T

In [18]:
data_with_distances_1 = data_with_distances_1\
  .withColumn('Time_O_tan_2', atan2(col('Time_O_sin'), col('Time_O_cos')))

In [20]:
from pyspark.sql import functions as F

# Define a window to partition by 'Date_EMG' and 'index_right', and order by 'ID'
window_spec = Window.partitionBy('Date_EMG', 'Time_O_tan_2', 'index_right').orderBy('ID')

# Flag the first occurrence of each 'ID' within each group
data_with_distances_1 = data_with_distances_1.withColumn(
  'is_first_occurrence', F.when(F.row_number().over(window_spec) == 1, 1).otherwise(0)
)

grouped = data_with_distances_1.groupBy(
  'Date_EMG', 
  'Time_O_tan_2', 
  'index_right'
  ).agg(
  first('Day_EMG'),
  first('Day_EMG_encoded'),
  first('Day_Type'),
  first('Day_Type_encoded'),
  first('Date_EMG_encoded'),
  avg('ID_Trip_Days'),
  mode('Type_Trip_OD'),
  mode('Type_Trip_OD_encoded'),
  first('Code_INSEE_O'),
  mode('Code_INSEE_D'),
  first('Zone_O'),
  mode('Zone_D'),
  first('Zone_O_encoded'),
  mode('Zone_D _encoded'),
  avg('Duration'),
  mode('Purpose_O'),
  mode('Purpose_D'),
  mode('Purpose_O_encoded'),
  mode('Purpose_D_encoded'),
  first('Time_O_Hour'),
  first('Time_D_Hour'),
  avg('temp'),
  avg('rhum'),
  avg('prcp'),
  avg('SPEED'),
  first('Time_O_sin'),
  first('Time_O_cos'),
  first('Time_D_sin'),
  first('Time_D_cos'),
  count('*').alias('Number of GPS points'),
  F.sum(
        F.when((F.col('is_first_occurrence') == 1) & (F.col('NB_CAR') > 0), F.col('Weight_Day')).otherwise(0)
    ).alias('Number of people having one or more car'),
  F.sum(
        F.when((F.col('is_first_occurrence') == 1) & (F.col('BIKE') > 0), F.col('Weight_Day')).otherwise(0)
    ).alias('Number of people having one or more BIKE'),
  F.sum(
        F.when((F.col('is_first_occurrence') == 1) & (F.col('ELECT_SCOOTER') > 0), F.col('Weight_Day')).otherwise(0)
    ).alias('Number of people having one or more ELECT_SCOOTER'),
  F.sum(
        F.when((F.col('is_first_occurrence') == 1) & (F.col('TWO_WHEELER') > 0), F.col('Weight_Day')).otherwise(0)
    ).alias('Number of people having one or more TWO_WHEELER'),
  sum(col('AGE')*col('Weight_Day'))/sum('Weight_Day'),
  sum('Weight_Day'),
  sum('distance_km'),
  first('fclass_encoded')
)

In [21]:
mode_pro_cat = data_with_distances_1\
  .groupBy('Date_EMG', 'Time_O_tan_2', 'index_right', 'PRO_CAT')\
  .sum('Weight_Day')

max_pro_cat_road_date = Window\
  .partitionBy('Date_EMG', 'Time_O_tan_2', 'index_right')\
  .orderBy(desc('sum(Weight_Day)'))

mode_pro_cat = mode_pro_cat.withColumn(
  'mode PRO_CAT', row_number().over(max_pro_cat_road_date)
).filter(col('mode PRO_CAT') == 1)\
.select('Date_EMG', 'Time_O_tan_2', 'index_right', 'PRO_CAT')

In [22]:
grouped_2 = grouped.join(mode_pro_cat, on=['Date_EMG', 'index_right', 'Time_O_tan_2'])

In [23]:
grouped_2.show(5)

+----------+-----------+------------------+--------------+----------------------+---------------+-----------------------+-----------------------+-----------------+------------------+--------------------------+-------------------+------------------+-------------+------------+---------------------+---------------------+-------------+---------------+---------------+-----------------------+-----------------------+------------------+------------------+------------------+---------+------------------+------------------+-------------------+-------------------+-------------------+-------------------+--------------------+---------------------------------------+----------------------------------------+-------------------------------------------------+-----------------------------------------------+-------------------------------------------+------------------+--------------------+---------------------+-------+
|  Date_EMG|index_right|      Time_O_tan_2|first(Day_EMG)|first(Day_EMG_encoded)|first(

In [24]:
grouped_2.filter(col('index_right') == '10202').show()

+----------+-----------+-------------------+--------------+----------------------+---------------+-----------------------+-----------------------+-----------------+------------------+--------------------------+-------------------+------------------+-------------+------------+---------------------+---------------------+-------------+---------------+---------------+-----------------------+-----------------------+------------------+------------------+------------------+---------+-----------------+------------------+-------------------+-------------------+-------------------+-------------------+--------------------+---------------------------------------+----------------------------------------+-------------------------------------------------+-----------------------------------------------+-------------------------------------------+---------------+-------------------+---------------------+-------+
|  Date_EMG|index_right|       Time_O_tan_2|first(Day_EMG)|first(Day_EMG_encoded)|first(Day

In [25]:
grouped_pandas = grouped_2.toPandas()

In [ ]:
import geopandas as gpd
roads = gpd.read_file('../data/france/gis_osm_roads_free_1.shp').to_crs(epsg=2154)

In [28]:
roads['index'] = roads.index.astype(str)

grouped_3_geo = roads[['index', 'geometry']].merge(grouped_pandas, left_on='index', right_on='index_right')

In [29]:
grouped_3_geo = grouped_3_geo.to_crs("EPSG:3857") # to meters
grouped_3_geo['length'] = grouped_3_geo.length

In [31]:
q33 = grouped_3_geo['sum(Weight_Day)'].quantile(1/3)
q66 = grouped_3_geo['sum(Weight_Day)'].quantile(2/3)

def label(row):
  if row['sum(Weight_Day)'] <= q33:
    return 0
  elif row['sum(Weight_Day)'] > q33 and row['sum(Weight_Day)'] <= q66:
    return 1
  
  return 2

grouped_3_geo['label'] = grouped_3_geo.apply(label, axis=1)

In [ ]:
grouped_3_geo\
  .drop(['geometry', 'index'], axis=1)\
  .to_csv('../data/model_features.csv', index=False)